# Discrete Latent Geometry Demo

This notebook demonstrates the discrete-latent geometry diagnostics for a trained VQ-family tokenizer. It is a lightweight wrapper around `scripts/analyze_discrete_latent_geometry.py`: use the CLI for canonical runs, and use this notebook to inspect summaries and figures.

The default paths point to local ignored `outputs/` artifacts. They do not reference private milestone paths. If artifacts are missing, run the tokenizer training and token extraction commands from the repository README or the verification notes first.

## Parameters

Set `RUN_ANALYSIS=True` to invoke the analysis script from the notebook. Leave it as `False` when the geometry outputs already exist and you only want to inspect plots.

In [ ]:
from pathlib import Path

# Standard VQ defaults. For RVQ q2, point these at the RVQ tokenizer/token outputs.
TOKENIZER_DIR = Path("outputs/sp500_vix_discrete/tokenizer/sp500_vix_causal_vq_tokenizer_seed0")
TOKEN_DATA_DIR = Path("outputs/sp500_vix_discrete/token_prior/tokens_codebook64_codebookdim16")
CONFIG_PATH = Path("configs/experiments/sp500_vix_causal_vq_tokenizer.yaml")
OUTPUT_DIR = Path("outputs/latent_geometry/sp500_vix_standard_vq")
BASE_DATA_DIR = Path("data/processed")
RUN_ANALYSIS = False

## Artifact Check

The notebook checks for tokenizer and token-data artifacts before running analysis. Missing artifacts are reported with explicit commands to run from the repository root.

In [ ]:
required_inputs = {
    "TOKENIZER_DIR": TOKENIZER_DIR,
    "TOKEN_DATA_DIR": TOKEN_DATA_DIR,
    "CONFIG_PATH": CONFIG_PATH,
    "BASE_DATA_DIR": BASE_DATA_DIR,
}
missing = {name: path for name, path in required_inputs.items() if not path.exists()}

if missing:
    print("Missing required artifacts for a real latent-geometry run:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    print("\nTo create standard VQ artifacts, run from the repository root:")
    print(
        "poetry run tcvae-train-tokenizer --config configs/experiments/sp500_vix_causal_vq_tokenizer.yaml --output-dir outputs/sp500_vix_discrete/tokenizer --base-data-dir data/processed --no-wandb"
    )
    print(
        "poetry run python scripts/extract_token_indices.py --config configs/experiments/sp500_vix_causal_vq_tokenizer.yaml --tokenizer-dir <tokenizer-dir> --output-dir outputs/sp500_vix_discrete/token_prior/tokens_codebook64_codebookdim16 --base-data-dir data/processed --seed 99"
    )
else:
    print("All configured inputs are present.")
    for name, path in required_inputs.items():
        print(f"  - {name}: {path}")

## Optional Analysis Run

When `RUN_ANALYSIS=True`, the notebook calls the same script used by the verification docs. W&B is intentionally not enabled from the notebook.

In [ ]:
import subprocess
import sys

analysis_command = [
    sys.executable,
    "scripts/analyze_discrete_latent_geometry.py",
    "--config",
    str(CONFIG_PATH),
    "--tokenizer-dir",
    str(TOKENIZER_DIR),
    "--token-data-dir",
    str(TOKEN_DATA_DIR),
    "--output-dir",
    str(OUTPUT_DIR),
    "--base-data-dir",
    str(BASE_DATA_DIR),
    "--plot-voronoi",
]

if RUN_ANALYSIS:
    if missing:
        print("RUN_ANALYSIS=True, but required inputs are missing. Resolve the paths above first.")
    else:
        print("Running:")
        print(" ".join(analysis_command))
        subprocess.run(analysis_command, check=True)
else:
    print("RUN_ANALYSIS=False; using any existing files under", OUTPUT_DIR)

## Numeric Summary

The JSON and Markdown files are the decision inputs. PNGs are for human inspection and W&B-style logging.

In [ ]:
import json

summary_path = OUTPUT_DIR / "codebook_geometry_summary.json"
markdown_path = OUTPUT_DIR / "latent_geometry_summary.md"

if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    usage = summary.get("usage", {})
    geometry = summary.get("geometry", {})
    metadata = summary.get("metadata", {})
    print("quantizer_type:", metadata.get("quantizer_type"))
    print("embedding_shape:", geometry.get("embedding_shape"))
    print("projection_method:", geometry.get("projection_method"))
    print("active_code_count:", usage.get("active_code_count"))
    print("perplexity:", usage.get("codebook_perplexity"))
    if usage.get("per_quantizer"):
        print("per_quantizer:")
        for item in usage["per_quantizer"]:
            print(
                "  q{quantizer_index}: active={active_code_count}, perplexity={codebook_perplexity:.6f}".format(
                    **item
                )
            )
else:
    print(f"No JSON summary found at {summary_path}")

if markdown_path.exists():
    print("\nMarkdown summary:")
    print(markdown_path.read_text()[:1200])
else:
    print(f"No Markdown summary found at {markdown_path}")

## Generated Plots

The cells below display plots if they exist. They do not embed plot files in the committed notebook after output stripping.

In [ ]:
from IPython.display import Image, Markdown, display

plot_names = [
    "codebook_projection.png",
    "codebook_usage_projection.png",
    "vix_bucket_code_usage.png",
    "token_trajectory_examples.png",
    "codebook_voronoi.png",
    "codebook_nearest_region.png",
    "q0_q1_pair_heatmap.png",
]

found_any = False
for plot_name in plot_names:
    plot_path = OUTPUT_DIR / plot_name
    if plot_path.exists():
        found_any = True
        display(Markdown(f"### `{plot_name}`"))
        display(Image(filename=str(plot_path)))

if not found_any:
    print(f"No expected plot files found under {OUTPUT_DIR}.")

## Interpretation

The current verification decision keeps standard VQ as the promoted public baseline: it has broad code use, VIX-sensitive usage structure, and a simple one-code interface for the additive scalar-conditioned causal AR prior.

RVQ q2 is useful as an ablation and future-work signal. Its q0/q1 geometry is interpretable, but the sparse same-time pair support makes token-prior generation harder. Unless the decision document is updated with stronger evidence, grouped tokenization and MGVQ should remain deferred or future work rather than the next implementation step.